In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession
        .builder
        .appName("spark-local-app")
        .master("local[*]")
        .getOrCreate()
)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/10/14 17:14:34 WARN Utils: Your hostname, DESKTOP-440A2SS, resolves to a loopback address: 127.0.1.1; using 172.20.131.26 instead (on interface eth0)
26/10/14 17:14:34 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/tem/de_lrn_prcts/.venv/lib/python3.14/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/10/14 17:14:46 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
employees = [
    (1, "Bob", "Tiger", 12432.343, "united States", "+1 2343434 34", "43 54 5465"),
    (2, "Alice", "Smith", 35000.50, "CANADA", "+1 555 123 4567", "123 Main St"),
    (3, "John", "Doe", 28900.12, "UK", "+44 7700 900123", "10 Downing St"),
    (4, "Maria", "Gonzalez", 42000.75, "MExiCO", "+52 55 1234 5678", "Av. Reforma 123"),
    (5, "Ken", "Tanaka", 39500.90, "Japan", "+81 3 1234 5678", "Tokyo, Chuo-ku"),
    (6, "Olga", "Petrova", 31000.25, "RUSSIa", "+7 999 123 45 67", "Moscow, Tverskaya 1"),
    (7, "Liam", "Brown", 27500.80, "Australia", "+61 2 1234 5678", "Sydney, George St"),
    (8, "Emma", "Lee", 45000.60, "Singapore", "+65 6123 4567", "Orchard Road 10"),
    (9, "Diego", "Rodriguez", 33000.40, "SPain", "+34 91 123 45 67", "Madrid, Gran Via"),
    (10, "Sophie", "Martin", 38000.30, "France", "+33 1 23 45 67 89", "Paris, Champs-Élysées")
]


In [5]:
df_employees = spark.createDataFrame(employees). \
    toDF("id", "first_name", "last_name", "salary", "country", "phone_number", "address")

In [6]:
df_employees.show()

+---+----------+---------+---------+-------------+-----------------+--------------------+
| id|first_name|last_name|   salary|      country|     phone_number|             address|
+---+----------+---------+---------+-------------+-----------------+--------------------+
|  1|       Bob|    Tiger|12432.343|united States|    +1 2343434 34|          43 54 5465|
|  2|     Alice|    Smith|  35000.5|       CANADA|  +1 555 123 4567|         123 Main St|
|  3|      John|      Doe| 28900.12|           UK|  +44 7700 900123|       10 Downing St|
|  4|     Maria| Gonzalez| 42000.75|       MExiCO| +52 55 1234 5678|     Av. Reforma 123|
|  5|       Ken|   Tanaka|  39500.9|        Japan|  +81 3 1234 5678|      Tokyo, Chuo-ku|
|  6|      Olga|  Petrova| 31000.25|       RUSSIa| +7 999 123 45 67| Moscow, Tverskaya 1|
|  7|      Liam|    Brown|  27500.8|    Australia|  +61 2 1234 5678|   Sydney, George St|
|  8|      Emma|      Lee|  45000.6|    Singapore|    +65 6123 4567|     Orchard Road 10|
|  9|     

In [12]:
from pyspark.sql.functions import col, initcap, length, lower, upper

df_employees. \
    select("id", "country"). \
    withColumn("country_lower", lower(col("country"))). \
    withColumn("country_upper", upper(col("country"))). \
    withColumn("country_initcap", initcap(col("country"))). \
    withColumn("country_length", length(col("country"))). \
    show()

+---+-------------+-------------+-------------+---------------+--------------+
| id|      country|country_lower|country_upper|country_initcap|country_length|
+---+-------------+-------------+-------------+---------------+--------------+
|  1|united States|united states|UNITED STATES|  United States|            13|
|  2|       CANADA|       canada|       CANADA|         Canada|             6|
|  3|           UK|           uk|           UK|             Uk|             2|
|  4|       MExiCO|       mexico|       MEXICO|         Mexico|             6|
|  5|        Japan|        japan|        JAPAN|          Japan|             5|
|  6|       RUSSIa|       russia|       RUSSIA|         Russia|             6|
|  7|    Australia|    australia|    AUSTRALIA|      Australia|             9|
|  8|    Singapore|    singapore|    SINGAPORE|      Singapore|             9|
|  9|        SPain|        spain|        SPAIN|          Spain|             5|
| 10|       France|       france|       FRANCE|     

In [19]:
from pyspark.sql.functions import col, substring

(
    df_employees
        .select("id", "first_name", "last_name",
                substring(col("phone_number"), -2, 4).cast("int").alias("last_digits"))
        .show()
)

+---+----------+---------+-----------+
| id|first_name|last_name|last_digits|
+---+----------+---------+-----------+
|  1|       Bob|    Tiger|         34|
|  2|     Alice|    Smith|         67|
|  3|      John|      Doe|         23|
|  4|     Maria| Gonzalez|         78|
|  5|       Ken|   Tanaka|         78|
|  6|      Olga|  Petrova|         67|
|  7|      Liam|    Brown|         78|
|  8|      Emma|      Lee|         67|
|  9|     Diego|Rodriguez|         67|
| 10|    Sophie|   Martin|         89|
+---+----------+---------+-----------+



In [20]:
(
    df_employees
        .select("id", "first_name", "last_name",
                substring(col("phone_number"), -2, 4).cast("int").alias("last_digits"))
        .printSchema()
)

root
 |-- id: long (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- last_digits: integer (nullable = true)



In [21]:
employees = [
    (1, "Bob", "Tiger", 12432.343, "united States", "+1 2343434 34,+67 2323 5434534", "43 54 5465"),
    (2, "Alice", "Smith", 35000.50, "CANADA", "+1 555 123 4567,+9993 32 4321", "123 Main St"),
    (3, "John", "Doe", 28900.12, "UK", "+44 7700 900123,+4 5344 3423 33", "10 Downing St"),
    (4, "Maria", "Gonzalez", 42000.75, "MExiCO", "+52 55 1234 5678,+23 434 54523", "Av. Reforma 123"),
    (5, "Ken", "Tanaka", 39500.90, "Japan", "+81 3 1234 5678,+2343 3453 2355", "Tokyo, Chuo-ku"),
    (6, "Olga", "Petrova", 31000.25, "RUSSIa", "+7 999 123 45 67,+2 345456 34334", "Moscow, Tverskaya 1"),
    (7, "Liam", "Brown", 27500.80, "Australia", "+61 2 1234 5678,+848 9938 5885", "Sydney, George St"),
    (8, "Emma", "Lee", 45000.60, "Singapore", "+65 6123 4567,+8494 49494 856856", "Orchard Road 10"),
    (9, "Diego", "Rodriguez", 33000.40, "SPain", "+34 91 123 45 67,+575 848347 238", "Madrid, Gran Via"),
    (10, "Sophie", "Martin", 38000.30, "France", "+33 1 23 45 67 89,+493 2923 5954", "Paris, Champs-Élysées")
]


In [23]:
df_employees = spark.createDataFrame(
    employees,
    schema="""
    id INT, first_name STRING, last_name STRING, salary DOUBLE,
    country STRING, phone_numbers STRING, address STRING
    """
)

In [ ]:
from pyspark.sql.functions import explode, split

df_employees.select("id", "phone_numbers", explode(split("phone_numbers", ",")).alias("phone")).show()

+---+--------------------+------------------+
| id|       phone_numbers|             phone|
+---+--------------------+------------------+
|  1|+1 2343434 34,+67...|     +1 2343434 34|
|  1|+1 2343434 34,+67...|  +67 2323 5434534|
|  2|+1 555 123 4567,+...|   +1 555 123 4567|
|  2|+1 555 123 4567,+...|     +9993 32 4321|
|  3|+44 7700 900123,+...|   +44 7700 900123|
|  3|+44 7700 900123,+...|   +4 5344 3423 33|
|  4|+52 55 1234 5678,...|  +52 55 1234 5678|
|  4|+52 55 1234 5678,...|     +23 434 54523|
|  5|+81 3 1234 5678,+...|   +81 3 1234 5678|
|  5|+81 3 1234 5678,+...|   +2343 3453 2355|
|  6|+7 999 123 45 67,...|  +7 999 123 45 67|
|  6|+7 999 123 45 67,...|   +2 345456 34334|
|  7|+61 2 1234 5678,+...|   +61 2 1234 5678|
|  7|+61 2 1234 5678,+...|    +848 9938 5885|
|  8|+65 6123 4567,+84...|     +65 6123 4567|
|  8|+65 6123 4567,+84...|+8494 49494 856856|
|  9|+34 91 123 45 67,...|  +34 91 123 45 67|
|  9|+34 91 123 45 67,...|   +575 848347 238|
| 10|+33 1 23 45 67 89...| +33 1 2

In [29]:
df_employees.select("id", "phone_numbers", split("phone_numbers", ",").alias("phone")).show(truncate=False)

+---+--------------------------------+-----------------------------------+
|id |phone_numbers                   |phone                              |
+---+--------------------------------+-----------------------------------+
|1  |+1 2343434 34,+67 2323 5434534  |[+1 2343434 34, +67 2323 5434534]  |
|2  |+1 555 123 4567,+9993 32 4321   |[+1 555 123 4567, +9993 32 4321]   |
|3  |+44 7700 900123,+4 5344 3423 33 |[+44 7700 900123, +4 5344 3423 33] |
|4  |+52 55 1234 5678,+23 434 54523  |[+52 55 1234 5678, +23 434 54523]  |
|5  |+81 3 1234 5678,+2343 3453 2355 |[+81 3 1234 5678, +2343 3453 2355] |
|6  |+7 999 123 45 67,+2 345456 34334|[+7 999 123 45 67, +2 345456 34334]|
|7  |+61 2 1234 5678,+848 9938 5885  |[+61 2 1234 5678, +848 9938 5885]  |
|8  |+65 6123 4567,+8494 49494 856856|[+65 6123 4567, +8494 49494 856856]|
|9  |+34 91 123 45 67,+575 848347 238|[+34 91 123 45 67, +575 848347 238]|
|10 |+33 1 23 45 67 89,+493 2923 5954|[+33 1 23 45 67 89, +493 2923 5954]|
+---+--------------------

In [34]:
df_employees. \
    select("id", "phone_numbers"). \
    withColumn("first_phone", split("phone_numbers", ",")[0]). \
    withColumn("second_phone", split("phone_numbers", ",")[1]). \
    show(truncate=False)

+---+--------------------------------+-----------------+------------------+
|id |phone_numbers                   |first_phone      |second_phone      |
+---+--------------------------------+-----------------+------------------+
|1  |+1 2343434 34,+67 2323 5434534  |+1 2343434 34    |+67 2323 5434534  |
|2  |+1 555 123 4567,+9993 32 4321   |+1 555 123 4567  |+9993 32 4321     |
|3  |+44 7700 900123,+4 5344 3423 33 |+44 7700 900123  |+4 5344 3423 33   |
|4  |+52 55 1234 5678,+23 434 54523  |+52 55 1234 5678 |+23 434 54523     |
|5  |+81 3 1234 5678,+2343 3453 2355 |+81 3 1234 5678  |+2343 3453 2355   |
|6  |+7 999 123 45 67,+2 345456 34334|+7 999 123 45 67 |+2 345456 34334   |
|7  |+61 2 1234 5678,+848 9938 5885  |+61 2 1234 5678  |+848 9938 5885    |
|8  |+65 6123 4567,+8494 49494 856856|+65 6123 4567    |+8494 49494 856856|
|9  |+34 91 123 45 67,+575 848347 238|+34 91 123 45 67 |+575 848347 238   |
|10 |+33 1 23 45 67 89,+493 2923 5954|+33 1 23 45 67 89|+493 2923 5954    |
+---+-------

In [35]:
spark.stop()